# pysnapgene-writer quickstart: edit pUC19

Read a real SnapGene map, add an annotation and a primer, and write it back out as
`pUC19_modified.dna`, with every colour, feature and note SnapGene put there intact.

```bash
conda env create -f ../environment.yml
conda activate pysnapgene-writer
pip install -e "..[dev]"
```

### Get pUC19.dna first

The map is not shipped with this repository. SnapGene's maps, notes and annotations are
copyrighted. They are free to use for academic, nonprofit and government users with the
citation *www.snapgene.com/resources*; commercial users need SnapGene's permission.

Download it from the
[SnapGene pUC19 page](https://www.snapgene.com/plasmids/basic_cloning_vectors/pUC19)
and save it as `examples/pUC19.dna`, next to this notebook.

In [1]:
from pathlib import Path

from Bio.SeqFeature import SeqFeature, SimpleLocation

import pysnapgene_writer
from pysnapgene_writer import Q_COLOR, Q_TRANSLATED, iter_packets, read_snapgene, write_snapgene

SOURCE = Path("pUC19.dna")
TARGET = Path("pUC19_modified.dna")

if not SOURCE.exists():
    raise FileNotFoundError(
        "pUC19.dna not found. Download it from "
        "https://www.snapgene.com/plasmids/basic_cloning_vectors/pUC19 "
        "and save it next to this notebook."
    )
print("pysnapgene_writer", pysnapgene_writer.__version__)

pysnapgene_writer 0.1.0


---
## 1. Read pUC19

In [2]:
record = read_snapgene(SOURCE)
seq = str(record.seq).upper()

print(f"{record.id}: {record.description}")
print(f"{len(seq)} bp, {record.annotations['topology']}, "
      f"double stranded={record.annotations['double_stranded']}")


def show(record):
    print(f"\n{'label':18} {'type':13} {'location':26} {'colour':8} translated")
    for f in record.features:
        print(f"{f.qualifiers['label'][0]:18} {f.type:13} {str(f.location):26} "
              f"{f.qualifiers.get(Q_COLOR, ['-'])[0]:8} "
              f"{'yes' if f.qualifiers.get(Q_TRANSLATED) else ''}")


show(record)

pUC19: pUC cloning vector.
2686 bp, circular, double stranded=True

label              type          location                   colour   translated
lacZα              CDS           [1748:2072](-)             #993366  yes
M13 rev            primer_bind   [2067:2084](-)             #a020f0  
M13 fwd            primer_bind   [1981:1998](+)             #a020f0  
MCS                misc_feature  [1998:2055]                #99ccff  
ori                rep_origin    join{[0:372](-), [2469:2686](-)} #ffff00  
lac operator       protein_bind  [2091:2108]                #31849b  
CAP binding site   protein_bind  [2160:2182]                #31849b  
lac promoter       promoter      join{[2140:2146](-), [2122:2140](-), [2115:2122](-)} #ffffff  
AmpR               CDS           join{[1334:1403](-), [542:1334](-)} #ccffcc  yes
AmpR promoter      promoter      [1403:1508](-)             #ffffff  
Amp-R              primer_bind   [1165:1185](+)             -        
L4440              primer_bind   [2

Each feature is an ordinary Biopython `SeqFeature`. SnapGene's display properties
come along as qualifiers: `snapgene_color`, `snapgene_translated`, and
`snapgene_native_type`. The `primer_bind` rows without a colour are SnapGene's primer
list. The coloured M13 sites are ordinary features.

---
## 2. Add an annotation

Mark the EcoRI…HindIII stretch of the multiple cloning site as the place an insert will
go. The positions are found from the sequence, not typed in. `strand=None` makes it a box
with no arrow, like SnapGene's own MCS feature; use `strand=1` or `-1` for an arrow.

In [3]:
ecori = seq.find("GAATTC")
hindiii = seq.find("AAGCTT")
start, end = sorted((ecori, hindiii))
end += 6  # include the whole of the second site

insert_site = SeqFeature(
    SimpleLocation(start, end, strand=None),
    type="misc_feature",
    qualifiers={
        "label": ["insert site"],
        "note": ["EcoRI to HindIII window of the pUC19 MCS"],
        Q_COLOR: ["#ff9ccd"],
    },
)
record.features.append(insert_site)
print(f"insert site: {start + 1}..{end} ({end - start} bp)")
print(seq[start:end])

insert site: 1999..2055 (57 bp)
GAATTCGAGCTCGGTACCCGGGGATCCTCTAGAGTCGACCTGCAGGCATGCAAGCTT


---
## 3. Add a sequencing primer

A 20-mer about 80 bp upstream of the insert site, on the top strand, so a read covers
the whole insert. Give a `primer_bind` feature a label and a location, and it goes into
SnapGene's Primers list, not the feature list. It is written the way SnapGene writes a
primer, with its binding site and melting temperature.

In [4]:
p_start = start - 80
p_end = p_start + 20
primer_seq = seq[p_start:p_end]

primer = SeqFeature(
    SimpleLocation(p_start, p_end, strand=1),
    type="primer_bind",
    qualifiers={"label": ["insert_seq_fwd"], "note": ["reads into the insert site"]},
)
record.features.append(primer)
print(f"insert_seq_fwd  5'-{primer_seq}-3'  {p_start + 1}..{p_end}")

insert_seq_fwd  5'-AAGGGGGATGTGCTGCAAGG-3'  1919..1938


---
## 4. Write `pUC19_modified.dna`

In [5]:
# SnapGene names a map after its file, so only the description needs changing
record.description = "pUC19 with the insert site and a sequencing primer annotated"

write_snapgene(record, TARGET)
print(f"wrote {TARGET} ({TARGET.stat().st_size:,} bytes)")

wrote pUC19_modified.dna (29,699 bytes)


Open it in SnapGene or the free SnapGene Viewer. The pink *insert site* box sits on the
MCS, and *insert_seq_fwd* is in the **Primers** view. Like pUC19's other primers it is not
drawn on the map until you turn on *Show Primers*. Everything else is exactly as SnapGene
saved it.

---
## 5. An unedited map is written back byte for byte

Reading a file and writing it straight back gives the same bytes, so SnapGene treats the
copy exactly like the original.

In [6]:
untouched = pysnapgene_writer.to_bytes(read_snapgene(SOURCE))
print(f"byte-identical copy: {untouched == SOURCE.read_bytes()}")

byte-identical copy: True


---
## 6. Check that nothing else changed

In [7]:
original = read_snapgene(SOURCE)
modified = read_snapgene(TARGET)


def summary(f):
    return (f.type, f.qualifiers["label"][0], str(f.location),
            f.qualifiers.get(Q_COLOR, [None])[0], f.qualifiers.get("translation"))


before = [summary(f) for f in original.features]
after = [summary(f) for f in modified.features]

print(f"sequence identical : {str(original.seq) == str(modified.seq)}")
print(f"features before    : {len(before)}, after: {len(after)}")
print(f"all originals kept : {all(f in after for f in before)}")
print("added              :", [s[1] for s in after if s not in before])

notes = next(p.data for p in iter_packets(TARGET.read_bytes()) if p.type == 0x06).decode()
print(f"\nreferences kept    : {'<References>' in notes}")
print(f"new description    : {'insert site and a sequencing primer' in notes}")
print(f"unparsed packets   : {len(original.annotations['snapgene_unknown_packets'])} read, "
      f"{len(modified.annotations['snapgene_unknown_packets'])} written back")

sequence identical : True
features before    : 20, after: 22
all originals kept : True
added              : ['insert site', 'insert_seq_fwd']

references kept    : True
new description    : True
unparsed packets   : 8 read, 8 written back


---
## 7. Look at the raw packets

A `.dna` file is a flat run of packets, each one a type byte, a length, and a payload.
Packets this library does not parse, such as enzyme sets and history, are written back
byte for byte.

In [8]:
for packet in iter_packets(TARGET.read_bytes()):
    print(f"0x{packet.type:02x}  {packet.name:28} {len(packet):>7,} bytes")

0x09  cookie                            14 bytes
0x00  DNA                            2,687 bytes
0x03  restriction enzymes            9,406 bytes
0x0b  history node                   2,766 bytes
0x07  history                        1,332 bytes
0x11  alignable sequences               45 bytes
0x08  additional sequence properties     289 bytes
0x0a  features                       5,856 bytes
0x05  primers                        6,074 bytes
0x06  notes                            728 bytes
0x0d  enzyme set                       345 bytes
0x0e  custom enzyme sets                41 bytes
0x1c  enzyme visibilities               51 bytes
